# Implementing a Custom GeoDataset

This notebook walks through how to implement your own geospatial dataset for use with DDR by subclassing `BaseGeoDataset`. The MERIT Hydro implementation is used as the concrete example throughout.

After reading this notebook you will know:
- How to build the adjacency zarr files your dataset needs using `ddr-engine`
- What data your dataset needs to provide and in what format
- What each abstract method must do, with the shape and semantic contracts spelled out
- The subtle invariants (unit conversions, tensor transpositions) that are easy to get wrong silently
- How to plug your custom class into a training or routing run

---
## 1. How the DataLoader and routing engine interact with your dataset

Before writing any code it helps to see where your dataset sits in the overall data flow.

```
TRAINING MODE
─────────────

  __init__
    └─ _init_training()          ← you implement this
         sets: self.gage_ids,
               self.observations,
               self.gages_adjacency,
               self.obs_reader

  DataLoader  →  collate_fn()    ← provided by BaseGeoDataset
                   │  calls dates.calculate_time_period()
                   └─ _collate_gages(batch)  ← you implement this
                        │  calls _build_common_tensors()
                        └─ RoutingDataclass
                               │
                               ▼
                          dmc (MC routing engine)


INFERENCE MODE  (testing / routing)
────────────────────────────────────

  __init__
    └─ _init_inference()         ← you implement this
         sets: self.routing_dataclass
               (calls _build_common_tensors() internally)

  DataLoader  →  collate_fn()    ← provided by BaseGeoDataset
                   │  reads self.routing_dataclass directly,
                   │  slices dates for each chunk
                   └─ RoutingDataclass
                               │
                               ▼
                          dmc (MC routing engine)
```

The `BaseGeoDataset` provides the PyTorch `Dataset` scaffolding (`__len__`, `__getitem__`, `collate_fn`). You implement the six abstract methods that fill in the domain-specific data.

---
## 2. What data your dataset must provide

| Data | Format | Purpose |
|---|---|---|
| **Catchment attributes** | `xr.Dataset` with a spatial coord (e.g. `COMID`) | Input features to the KAN |
| **Flowpath attributes** | `gpd.GeoDataFrame` indexed by catchment ID | Physical channel geometry for MC routing |
| **CONUS adjacency** | zarr group — built by `ddr-engine` | Full directed flow network |
| **Gages adjacency** *(training)* | zarr group — built by `ddr-engine` | Pre-built per-gage subgraphs |
| **Streamflow observations** *(training/testing)* | `xr.Dataset` from `IcechunkUSGSReader` | Training targets |

The `data_sources` section of your config points to all of these files. **The two adjacency zarr files are not provided by the raw hydrofabric** — you build them yourself with `ddr-engine` before training. Section 3 explains how.

### Flowpath columns your dataset must supply

| Column | Units | Required? | Notes |
|---|---|---|---|
| Reach length | **metres** | Yes | MERIT stores `lengthkm`; multiply by 1000 |
| Channel slope | dimensionless | Yes | |
| Muskingum *x* | 0–0.5 | Yes | Lynker reads per-reach; MERIT uses fixed 0.3 |
| Top width | metres | No | Set `torch.empty(0)` if using Leopold & Maddock geometry |
| Side slope *z* | H:V | No | Set `torch.empty(0)` if using Leopold & Maddock geometry |

---
## 3. Building adjacency matrices with `ddr-engine`

The two zarr files (`conus_adjacency` and `gages_adjacency`) encode the directed flow network as sparse matrices. They do not come from the raw hydrofabric — you build them once using the `ddr-engine` workspace package, which wraps [RustWorkX](https://www.rustworkx.org/) to construct and topologically sort the river network graph.

```
Raw network geometry
(shapefile or geopackage)
         │
         ▼
    ddr-engine
    ├── reads connectivity columns (NextDownID / toid)
    ├── builds a RustWorkX PyDiGraph
    ├── detects and removes cycles
    ├── computes topological sort  (rx.topological_sort)
    └── writes binsparse COO zarr
         │
    ┌────┴────────────────────────────────┐
    ▼                                     ▼
conus_adjacency.zarr            gages_adjacency.zarr
(full network,                  (one subgroup per gage,
 ~866k × 866k for CONUS)         upstream segments only)
         │
         └──────────────────┐
                            ▼
                  BaseGeoDataset.__init__
                  reads both via read_zarr()
```

### 3.1 Install

`ddr-engine` is a workspace package — it is installed automatically alongside `ddr`:

```bash
uv sync --all-packages
```

### 3.2 Required input geometry

#### MERIT Hydro — shapefile

The engine reads the standard MERIT Hydro flowlines shapefile. The required columns are:

| Column | Type | Description |
|---|---|---|
| `COMID` | int | Unique reach identifier |
| `NextDownID` | int | COMID of the immediately downstream reach (0 = terminal) |
| `up1`–`up4` | int | COMIDs of up to four upstream tributaries (0 = none) |

#### Lynker Hydrofabric v2.2 — GeoPackage

The engine queries two tables from the SQLite GeoPackage:

| Table | Columns used | Description |
|---|---|---|
| `flowpaths` | `id`, `toid`, `tot_drainage_areasqkm` | Reach IDs (`wb-*`) and their downstream nexus |
| `network` | `id`, `toid`, `hl_uri` | Nexus-to-reach connections and gage links |

### 3.3 Build commands

Run one of these once before training. Both write two zarr files to `--path`.

```bash
# MERIT Hydro
uv run python -m ddr_engine.merit /path/to/riv_pfaf_X_MERIT_Hydro_v07_Basins_v01_bugfix1.shp \
    --path data/ \
    --gages references/gage_info/dhbv2_gages.csv

# Lynker Hydrofabric v2.2
uv run python -m ddr_engine.lynker_hydrofabric /path/to/conus_nextgen.gpkg \
    --path data/ \
    --gages references/gage_info/dhbv2_gages.csv
```

Output files:

```
data/
├── merit_conus_adjacency.zarr          ← set as data_sources.conus_adjacency
└── merit_gages_conus_adjacency.zarr    ← set as data_sources.gages_adjacency
```

### 3.4 What the zarr files contain (binsparse COO format)

Both zarr files use the same binsparse COO format. Each group contains four arrays and a set of metadata attributes.

**Arrays:**

| Array | dtype | Shape | Meaning |
|---|---|---|---|
| `indices_0` | int32 | `(nnz,)` | Row indices — *downstream* segment positions |
| `indices_1` | int32 | `(nnz,)` | Column indices — *upstream* segment positions |
| `values` | uint8 | `(nnz,)` | All 1 (adjacency matrix) |
| `order` | int32 | `(N,)` | Domain IDs in topological sort order (COMIDs for MERIT, numeric portion of `wb-*` for Lynker) |

**Attributes (CONUS file):**

| Key | Value |
|---|---|
| `format` | `"COO"` |
| `shape` | `[N, N]` |
| `geodataset` | `"merit"` or `"lynker"` |

**Attributes (per-gage subgroup in gages file):**

| Key | Value |
|---|---|
| `gage_catchment` | The origin segment ID (COMID or `wb-*`) for this gage |
| `gage_idx` | The integer index of the origin in the CONUS `order` array |

The matrix is **lower triangular**: `A[i, j] = 1` means segment `j` flows into segment `i`. Topological order guarantees that `j < i` always holds, so upstream segments always have lower indices.

```
     0  1  2  3  4   ← upstream
   ┌───────────────┐
 0 │ .             │
 1 │ 1  .          │  A[1,0]=1: segment 0 → segment 1
 2 │ .  1  .       │  A[2,1]=1: segment 1 → segment 2
 3 │ .  .  1  .    │
 4 │ .  .  1  1  . │  A[4,2]=1 and A[4,3]=1: confluence
   └───────────────┘
   ↑ downstream
```

### 3.5 How your `__init__` loads the zarr files

In `__init__` you load both zarr files using `read_zarr` from `ddr.io.readers` and store them as instance attributes. The CONUS file is read unconditionally; the gages file is only needed for training and gage-mode inference.

In [ ]:
# Illustrative — this goes inside your __init__
from pathlib import Path

from ddr.io.readers import read_zarr

# Load the full CONUS network (always needed)
self.conus_adjacency = read_zarr(Path(cfg.data_sources.conus_adjacency))

# The 'order' array gives you domain IDs in topological sort order.
# For MERIT these are integer COMIDs; for Lynker they are the numeric
# portion of 'wb-*' IDs stored as int32.
self.merit_ids = self.conus_adjacency["order"][:]  # shape (N,)

# During _init_training / _init_inference (gage mode) you also load:
self.gages_adjacency = read_zarr(Path(cfg.data_sources.gages_adjacency))
# self.gages_adjacency["01570500"]            ← subgroup keyed by STAID
# self.gages_adjacency["01570500"]["indices_0"]   indices of upstream rows
# self.gages_adjacency["01570500"].attrs["gage_idx"]  position in CONUS order

### 3.6 How `_collate_gages` and inference use the zarr files

Two functions in `ddr.io.builders` do all the heavy graph work so you don't have to replicate it:

- **`construct_network_matrix(batch, gages_adjacency)`** — merges the per-gage COO subsets for all gages in a mini-batch into a single compressed COO matrix and returns the gage-origin indices.
- **`_build_network_graph(conus_adjacency)`** — reconstructs the full RustWorkX `PyDiGraph` from the CONUS zarr, used only for the `target_catchments` inference branch to find all ancestors of a target segment via `rx.ancestors()`.

You call `construct_network_matrix` inside `_collate_gages` and `_build_network_graph` inside `_init_inference` (target-catchments branch). You do not need to read zarr arrays directly in these methods.

---
## 4. Imports and the abstract interface

The cell below shows the imports your subclass will need.

In [ ]:
from pathlib import Path
from typing import Any

import geopandas as gpd
import numpy as np
import torch
import xarray as xr
from scipy import sparse

from ddr.geodatazoo.base_geodataset import BaseGeoDataset
from ddr.geodatazoo.dataclasses import Dates, RoutingDataclass
from ddr.io.builders import construct_network_matrix, create_hydrofabric_observations
from ddr.io.readers import (
    IcechunkUSGSReader,
    build_flow_scale_tensor,
    fill_nans,
    filter_headwater_gages,
    naninfmean,
    read_zarr,
)
from ddr.io.statistics import set_statistics
from ddr.validation.configs import Config
from ddr.validation.enums import Mode

---
## 5. Step-by-step: the six abstract methods

We walk through each method in the order it is called, using the real MERIT implementation.

### 5.1 `__init__` — constructor pattern

`BaseGeoDataset` does not define `__init__`, so yours is the only one. The pattern used by both built-in datasets is:

1. Store the config and build a `Dates` object.
2. Declare all instance attributes as `None` (mypy requires this).
3. Call `_load_attributes()` and compute normalisation statistics.
4. Load the flowpath geometry.
5. Load the CONUS adjacency zarr (built by `ddr-engine` in step 3 above).
6. Delegate to `_init_training()` or `_init_inference()` based on `cfg.mode`.

Steps 1–5 run for every mode, so keep them cheap. The per-mode initialisation in step 6 is where expensive observation loading or graph traversal happens.

In [ ]:
# Constructor pattern — illustrative skeleton, not executable on its own


class MyDataset(BaseGeoDataset):
    """Custom GeoDataset implementation — replace this docstring with a description of your dataset."""

    def __init__(self, cfg: Config) -> None:
        self.cfg = cfg
        self.dates = Dates(**self.cfg.experiment.model_dump())

        # Declare optional attributes so type-checkers are happy
        self.gage_ids: np.ndarray | None = None
        self.routing_dataclass: RoutingDataclass | None = None
        self.observations: Any = None
        self.gages_adjacency: Any = None

        # Load attributes and compute statistics (used for normalisation)
        self.attribute_ds = self._load_attributes()
        self.attr_stats = set_statistics(self.cfg, self.attribute_ds)
        self.attributes_list = list(self.cfg.kan.input_var_names)

        # Pre-build mean/std tensors so normalisation is a single broadcast op
        self.means = torch.tensor(
            [self.attr_stats[attr].iloc[2] for attr in self.attributes_list],
            device=self.cfg.device,
            dtype=torch.float32,
        ).unsqueeze(1)  # shape (num_attrs, 1) for broadcasting over N catchments
        self.stds = torch.tensor(
            [self.attr_stats[attr].iloc[3] for attr in self.attributes_list],
            device=self.cfg.device,
            dtype=torch.float32,
        ).unsqueeze(1)

        # Load the CONUS adjacency zarr built by ddr-engine
        self.conus_adjacency = read_zarr(Path(cfg.data_sources.conus_adjacency))
        self.merit_ids = self.conus_adjacency["order"][:]  # domain IDs in topo order

        if cfg.mode == Mode.TRAINING:
            self._init_training()
        else:
            self._init_inference()

### 5.2 `_load_attributes` — load your attribute store

**Contract:** return an `xr.Dataset` whose data variables cover every name in `cfg.kan.input_var_names`, with a spatial coordinate that you can use to select individual catchments.

MERIT stores attributes in a NetCDF file indexed by `COMID`. Lynker uses an icechunk store indexed by `divide_id`. The format does not matter as long as the result is an `xr.Dataset`.

In [ ]:
# MERIT implementation
def _load_attributes(self) -> xr.Dataset:
    # xr.open_mfdataset handles both single files and glob patterns
    return xr.open_mfdataset(self.cfg.data_sources.attributes)

### 5.3 `_get_attributes` — fetch attributes for a batch of catchment IDs

**Output shape: `(num_attributes, N)` — attributes are rows, catchments are columns.**

This feels backwards compared to the usual ML convention of `(N, features)`, but it makes the per-attribute NaN-filling and normalisation arithmetic simpler (a single row operation rather than a column operation).

Two things to handle carefully:
- **Missing catchments**: some IDs in `catchment_ids` may not be present in your attribute store (edge segments, cross-boundary reaches). Pre-fill the output tensor with `np.nan` and write a validity mask — `fill_nans` will replace them with per-attribute means.
- **ID type**: MERIT COMIDs are integers; Lynker divide IDs are strings. Ensure your `id_to_index` lookup dict uses the same type as the incoming `catchment_ids`.

In [ ]:
# MERIT implementation
def _get_attributes(
    self,
    catchment_ids: np.ndarray,
    device: str | torch.device = "cpu",
    dtype: torch.dtype = torch.float32,
) -> torch.Tensor:
    valid_indices = []  # positions in the attribute store
    divide_idx_mask = []  # positions in the output tensor where data will be written

    for i, divide_id in enumerate(catchment_ids):
        comid = int(divide_id)  # MERIT IDs are integers
        if comid in self.id_to_index:
            valid_indices.append(self.id_to_index[comid])
            divide_idx_mask.append(i)
        # Missing IDs are silently skipped; their column stays NaN

    assert valid_indices, "No valid COMIDs found in this batch"

    # Pre-fill with NaN so fill_nans can substitute per-attribute means later
    output = torch.full(
        (len(self.attributes_list), len(catchment_ids)),
        np.nan,
        device=device,
        dtype=dtype,
    )  # shape: (num_attributes, N)

    _ds = self.attribute_ds[self.attributes_list].isel(COMID=valid_indices).compute()
    data_tensor = torch.from_numpy(
        _ds.to_array(dim="COMID").values  # also (num_attributes, N_valid)
    ).to(device=device, dtype=dtype)

    output[:, divide_idx_mask] = data_tensor
    return fill_nans(attr=output, row_means=self.means)

### 5.4 `_build_common_tensors` — the core tensor assembly step

This method is called by `_collate_gages`, `_build_routing_data_gages`, `_build_routing_data_target_catchments`, and `_build_routing_data_all_catchments`. All four paths need the same four outputs, so the logic is centralised here.

The input `csr_matrix` is a *compressed* scipy CSR for the current batch's subgraph — it was assembled from the zarr COO data by `construct_network_matrix` (training) or by filtering the CONUS COO directly (inference). By the time `_build_common_tensors` is called, the graph-level zarr work is already done.

**Return values:**

| Variable | Shape | Notes |
|---|---|---|
| `adjacency_matrix` | `(N, N)` sparse CSR | On `cfg.device` |
| `spatial_attributes` | `(num_attributes, N)` | Raw, NaNs filled |
| `normalized_spatial_attributes` | `(N, num_attributes)` | Z-score normalised, **transposed** |
| `flowpath_tensors` | `dict[str, Tensor]` | Keys: `length`, `slope`, `x`, `top_width`, `side_slope` |

> **Why the transposition?** `spatial_attributes` is `(num_attrs, N)` because that layout makes per-attribute operations like `(attr - mean) / std` a single broadcast. But the KAN expects `(N, num_features)`. The transpose in `_build_common_tensors` is the only place this conversion happens — do not add it elsewhere.

> **`length` units:** the MC engine expects **metres**. MERIT stores reach length in kilometres (`lengthkm`), so you must multiply by 1000. Lynker already stores `Length_m` in metres.

> **Unused geometry tensors:** if your dataset uses Leopold & Maddock power-law geometry (MERIT), set `top_width` and `side_slope` to `torch.empty(0)`. The routing engine checks the tensor size to decide which geometry branch to use.

In [ ]:
# MERIT implementation
def _build_common_tensors(
    self,
    csr_matrix: sparse.csr_matrix,  # (N, N) scipy CSR for the subgraph
    catchment_ids: np.ndarray,  # (N,) IDs in compressed-index order
    flowpath_attr: gpd.GeoDataFrame,  # indexed by catchment ID
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, dict[str, torch.Tensor]]:

    # 1. Adjacency matrix — convert scipy CSR to a PyTorch sparse CSR tensor
    adjacency_matrix = torch.sparse_csr_tensor(
        crow_indices=csr_matrix.indptr,
        col_indices=csr_matrix.indices,
        values=csr_matrix.data,
        size=csr_matrix.shape,
        device=self.cfg.device,
        dtype=torch.float32,
    )  # shape (N, N)

    # 2. Raw attributes — shape (num_attributes, N)
    spatial_attributes = self._get_attributes(
        catchment_ids=catchment_ids,
        device=self.cfg.device,
        dtype=torch.float32,
    )

    # Fill any remaining per-row NaNs with the batch mean
    for r in range(spatial_attributes.shape[0]):
        row_means = torch.nanmean(spatial_attributes[r])
        nan_mask = torch.isnan(spatial_attributes[r])
        spatial_attributes[r, nan_mask] = row_means

    # 3. Z-score normalise then TRANSPOSE from (num_attrs, N) → (N, num_attrs)
    #    self.means and self.stds are both (num_attrs, 1), so the broadcast
    #    operates over the N dimension automatically.
    normalized_spatial_attributes = (spatial_attributes - self.means) / self.stds
    normalized_spatial_attributes = normalized_spatial_attributes.T  # (N, num_attrs)

    # 4. Physical channel properties — each (N,)
    flowpath_tensors = {
        # MERIT stores km; the MC engine requires metres — multiply by 1000
        "length": fill_nans(
            torch.tensor(flowpath_attr["lengthkm"].values, dtype=torch.float32),
            row_means=self.phys_means[0],
        )
        * 1000,
        "slope": fill_nans(
            torch.tensor(flowpath_attr["slope"].values, dtype=torch.float32),
            row_means=self.phys_means[1],
        ),
        # MERIT uses a fixed Muskingum x = 0.3 for all reaches
        "x": torch.full_like(
            torch.tensor(flowpath_attr["lengthkm"].values, dtype=torch.float32),
            fill_value=0.3,
        ),
        # Leopold & Maddock geometry — top_width and side_slope are derived
        # from the learned parameters, so pass empty tensors here
        "top_width": torch.empty(0),
        "side_slope": torch.empty(0),
    }

    return adjacency_matrix, spatial_attributes, normalized_spatial_attributes, flowpath_tensors

### 5.5 `_init_training` — set the four required instance attributes

The base class `collate_fn` reads `self.gage_ids` directly (in `__len__` and `__getitem__`), and `_collate_gages` reads `self.gages_adjacency` and `self.obs_reader`. None of these are set by the base class — they are a hidden contract that `_init_training` must fulfil.

**Required attributes:**

| Attribute | Type | Used by |
|---|---|---|
| `self.gage_ids` | `np.ndarray` | `__len__`, `__getitem__`, DataLoader sampling |
| `self.observations` | loaded observation dataset | `_collate_gages` → `create_hydrofabric_observations` |
| `self.gages_adjacency` | zarr group (from `ddr-engine`) | `_collate_gages` → `construct_network_matrix` |
| `self.obs_reader` | `IcechunkUSGSReader` | `_collate_gages` → `build_flow_scale_tensor` (needs `gage_dict`) |

In [ ]:
# MERIT implementation
def _init_training(self) -> None:
    if self.cfg.data_sources.gages is None or self.cfg.data_sources.gages_adjacency is None:
        raise ValueError("Training requires gages and gages_adjacency to be defined")

    # Load observations and build gage list from the store
    self.obs_reader = IcechunkUSGSReader(cfg=self.cfg)
    self.observations = self.obs_reader.read_data(dates=self.dates)
    self.gage_ids = np.array([str(_id.zfill(8)) for _id in self.obs_reader.gage_dict["STAID"]])

    # Load the per-gage adjacency zarr built by ddr-engine
    self.gages_adjacency = read_zarr(Path(self.cfg.data_sources.gages_adjacency))

    # Remove headwater gages (no upstream reaches → routing is trivial)
    self.gage_ids, n_removed = filter_headwater_gages(self.gage_ids, self.gages_adjacency)
    print(f"Training mode: {len(self.gage_ids)} gauged locations ({n_removed} headwaters removed)")

### 5.6 `_init_inference` — build `self.routing_dataclass`

During inference the entire routing domain is built once in `__init__` and stored as `self.routing_dataclass`. The base class `collate_fn` reads it on every iteration, updating only the time window.

Three sub-cases must be handled in priority order:

1. **`target_catchments`** — user has specified explicit segment IDs; use `_build_network_graph` to rebuild the RustWorkX DAG from the CONUS zarr, then call `rx.ancestors()` to find all upstream segments.
2. **`gages` mode** — user has a gage CSV and gages adjacency zarr; build the union of gage subgraphs.
3. **All segments** — fallback; assemble the full CONUS COO from `indices_0`/`indices_1`.

In [ ]:
# MERIT implementation — abbreviated to show the three-branch structure
from ddr.io.builders import _build_network_graph


def _init_inference(self) -> None:
    if self.cfg.data_sources.target_catchments is not None:
        # Branch 1: explicit target catchments
        # Rebuilds the RustWorkX DAG from the CONUS zarr so we can call
        # rx.ancestors() to find every segment upstream of each target.
        self.network_graph, self.hf_id_to_node, _ = _build_network_graph(self.conus_adjacency)
        self.routing_dataclass = self._build_routing_data_target_catchments()

    elif self.cfg.data_sources.gages is not None and self.cfg.data_sources.gages_adjacency is not None:
        # Branch 2: gage mode — load observations so metrics can be computed
        self.obs_reader = IcechunkUSGSReader(cfg=self.cfg)
        self.observations = self.obs_reader.read_data(dates=self.dates)
        self.gage_ids = np.array([str(_id.zfill(8)) for _id in self.obs_reader.gage_dict["STAID"]])
        self.gages_adjacency = read_zarr(Path(self.cfg.data_sources.gages_adjacency))
        self.gage_ids, _ = filter_headwater_gages(self.gage_ids, self.gages_adjacency)
        self.routing_dataclass = self._build_routing_data_gages()

    else:
        # Branch 3: route every segment — reads indices_0/indices_1 directly
        # from the CONUS zarr to build the full CSR matrix
        self.routing_dataclass = self._build_routing_data_all_catchments()

### 5.7 `_collate_gages` — build a RoutingDataclass for a training mini-batch

This is the most complex method. It is called once per mini-batch during training. The batch arrives as an array of gage IDs; you must:

1. Call `construct_network_matrix(batch, self.gages_adjacency)` — this reads the per-gage COO subsets from the gages zarr and merges them into a single batch-level COO matrix.
2. Remap the global CONUS indices to a 0-based compressed index space.
3. Call `_build_common_tensors` to get the tensors.
4. Build `outflow_idx` — the ragged list that tells the routing engine where each gage outlet is.
5. Fetch observations aligned to this batch's gages and time window.
6. Return a fully-populated `RoutingDataclass`.

In [ ]:
# MERIT implementation
def _collate_gages(self, batch: np.ndarray) -> RoutingDataclass:
    # Drop gages not in the adjacency structure (e.g. cross-boundary)
    valid_mask = np.isin(batch, list(self.gages_adjacency.keys()))
    batch = batch[valid_mask].tolist()

    # construct_network_matrix reads the per-gage COO subsets from the
    # gages zarr and merges them.  Returns:
    #   coo        — merged COO in CONUS index space
    #   _gage_idx  — CONUS index of each gage's origin segment
    #   gage_catchment — origin segment IDs
    coo, _gage_idx, gage_catchment = construct_network_matrix(batch, self.gages_adjacency)

    # Identify which global indices are active in this subgraph
    edge_indices = np.unique(np.concatenate([coo.row, coo.col])) if coo.nnz > 0 else np.array([], dtype=int)
    gage_indices = np.array(_gage_idx, dtype=int)
    active_indices = np.unique(np.concatenate([edge_indices, gage_indices]))

    # Remap CONUS global indices → 0-based compressed indices
    index_mapping = {orig: comp for comp, orig in enumerate(active_indices)}

    if coo.nnz > 0:
        c_rows = np.array([index_mapping[i] for i in coo.row])
        c_cols = np.array([index_mapping[i] for i in coo.col])
    else:
        c_rows = c_cols = np.array([], dtype=int)

    N = len(active_indices)
    compressed_csr = sparse.coo_matrix((coo.data[: len(c_rows)], (c_rows, c_cols)), shape=(N, N)).tocsr()

    # Look up domain IDs and flowpath geometry for compressed segments
    compressed_ids = self.merit_ids[active_indices]  # from conus_adjacency["order"]
    compressed_flowpath = self.flowpath_attr.reindex(compressed_ids)

    # outflow_idx[i] = compressed column indices that drain into gage i
    outflow_idx = []
    for _idx in _gage_idx:
        mask = np.isin(coo.row, _idx)
        orig_cols = coo.col[np.where(mask)[0]]
        if len(orig_cols) > 0:
            outflow_idx.append(np.array([index_mapping[c] for c in orig_cols]))
        else:
            outflow_idx.append(np.array([index_mapping[int(_idx)]]))

    gage_compressed_indices = [index_mapping[int(i)] for i in _gage_idx]
    flow_scale = build_flow_scale_tensor(
        batch=batch,
        gage_dict=self.obs_reader.gage_dict,
        gage_compressed_indices=gage_compressed_indices,
        num_segments=N,
    )

    adjacency_matrix, spatial_attributes, normalized_spatial_attributes, flowpath_tensors = (
        self._build_common_tensors(compressed_csr, compressed_ids, compressed_flowpath)
    )

    observations = create_hydrofabric_observations(
        dates=self.dates,
        gage_ids=np.array(batch),
        observations=self.observations,
    )

    return RoutingDataclass(
        spatial_attributes=spatial_attributes,
        length=flowpath_tensors["length"],
        slope=flowpath_tensors["slope"],
        side_slope=flowpath_tensors["side_slope"],
        top_width=flowpath_tensors["top_width"],
        x=flowpath_tensors["x"],
        dates=self.dates,
        adjacency_matrix=adjacency_matrix,
        normalized_spatial_attributes=normalized_spatial_attributes,
        observations=observations,
        divide_ids=compressed_ids,
        outflow_idx=outflow_idx,
        gage_catchment=gage_catchment,
        flow_scale=flow_scale,
    )

---
## 6. The complete class

The cell below assembles all six methods into the complete MERIT implementation as it appears in `src/ddr/geodatazoo/merit.py`.

In [ ]:
import logging

import rustworkx as rx

from ddr.io.builders import _build_network_graph
from ddr.io.readers import filter_gages_by_area_threshold, filter_gages_by_da_valid

log = logging.getLogger(__name__)


class Merit(BaseGeoDataset):
    """BaseGeoDataset implementation for the MERIT-Hydro global river network."""

    def __init__(self, cfg: Config) -> None:
        self.cfg = cfg
        self.dates = Dates(**self.cfg.experiment.model_dump())
        self.gage_ids: np.ndarray | None = None
        self.routing_dataclass: RoutingDataclass | None = None
        self.network_graph: rx.PyDiGraph | None = None
        self.hf_id_to_node: dict[int, int] | None = None
        self.observations: Any = None
        self.gages_adjacency: Any = None
        self.target_catchments: list[str] | None = None

        self.attribute_ds = self._load_attributes()
        self.attr_stats = set_statistics(self.cfg, self.attribute_ds)
        self.id_to_index = {comid: idx for idx, comid in enumerate(self.attribute_ds.COMID.values)}
        self.attributes_list = list(self.cfg.kan.input_var_names)

        self.means = torch.tensor(
            [self.attr_stats[attr].iloc[2] for attr in self.attributes_list],
            device=self.cfg.device,
            dtype=torch.float32,
        ).unsqueeze(1)
        self.stds = torch.tensor(
            [self.attr_stats[attr].iloc[3] for attr in self.attributes_list],
            device=self.cfg.device,
            dtype=torch.float32,
        ).unsqueeze(1)

        _flowpath_attr = gpd.read_file(self.cfg.data_sources.geospatial_fabric_gpkg).set_index("COMID")
        self.flowpath_attr = _flowpath_attr[~_flowpath_attr.index.duplicated(keep="first")]
        self.phys_means = torch.tensor(
            [naninfmean(self.flowpath_attr[attr].values) for attr in ["lengthkm", "slope"]],
            device=self.cfg.device,
            dtype=torch.float32,
        ).unsqueeze(1)

        self.conus_adjacency = read_zarr(Path(cfg.data_sources.conus_adjacency))
        self.merit_ids = self.conus_adjacency["order"][:]  # COMIDs in topo order

        if cfg.mode == Mode.TRAINING:
            self._init_training()
        else:
            self._init_inference()

    def _load_attributes(self) -> xr.Dataset:
        return xr.open_mfdataset(self.cfg.data_sources.attributes)

    def _get_attributes(
        self,
        catchment_ids: np.ndarray,
        device: str | torch.device = "cpu",
        dtype: torch.dtype = torch.float32,
    ) -> torch.Tensor:
        valid_indices, divide_idx_mask = [], []
        for i, divide_id in enumerate(catchment_ids):
            comid = int(divide_id)
            if comid in self.id_to_index:
                valid_indices.append(self.id_to_index[comid])
                divide_idx_mask.append(i)
        assert valid_indices, "No valid COMIDs found in this batch"
        output = torch.full(
            (len(self.attributes_list), len(catchment_ids)), np.nan, device=device, dtype=dtype
        )
        _ds = self.attribute_ds[self.attributes_list].isel(COMID=valid_indices).compute()
        data_tensor = torch.from_numpy(_ds.to_array(dim="COMID").values).to(device=device, dtype=dtype)
        output[:, divide_idx_mask] = data_tensor
        return fill_nans(attr=output, row_means=self.means)

    def _build_common_tensors(
        self,
        csr_matrix: sparse.csr_matrix,
        catchment_ids: np.ndarray,
        flowpath_attr: gpd.GeoDataFrame,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, dict[str, torch.Tensor]]:
        adjacency_matrix = torch.sparse_csr_tensor(
            crow_indices=csr_matrix.indptr,
            col_indices=csr_matrix.indices,
            values=csr_matrix.data,
            size=csr_matrix.shape,
            device=self.cfg.device,
            dtype=torch.float32,
        )
        spatial_attributes = self._get_attributes(
            catchment_ids=catchment_ids, device=self.cfg.device, dtype=torch.float32
        )
        for r in range(spatial_attributes.shape[0]):
            nan_mask = torch.isnan(spatial_attributes[r])
            spatial_attributes[r, nan_mask] = torch.nanmean(spatial_attributes[r])
        normalized_spatial_attributes = ((spatial_attributes - self.means) / self.stds).T
        flowpath_tensors = {
            "length": fill_nans(
                torch.tensor(flowpath_attr["lengthkm"].values, dtype=torch.float32),
                row_means=self.phys_means[0],
            )
            * 1000,
            "slope": fill_nans(
                torch.tensor(flowpath_attr["slope"].values, dtype=torch.float32),
                row_means=self.phys_means[1],
            ),
            "x": torch.full_like(
                torch.tensor(flowpath_attr["lengthkm"].values, dtype=torch.float32), fill_value=0.3
            ),
            "top_width": torch.empty(0),
            "side_slope": torch.empty(0),
        }
        return adjacency_matrix, spatial_attributes, normalized_spatial_attributes, flowpath_tensors

    def _init_training(self) -> None:
        if self.cfg.data_sources.gages is None or self.cfg.data_sources.gages_adjacency is None:
            raise ValueError("Training requires gages and gages_adjacency")
        self.obs_reader = IcechunkUSGSReader(cfg=self.cfg)
        self.observations = self.obs_reader.read_data(dates=self.dates)
        self.gage_ids = np.array([str(_id.zfill(8)) for _id in self.obs_reader.gage_dict["STAID"]])
        if "DA_VALID" in self.obs_reader.gage_dict:
            self.gage_ids, n = filter_gages_by_da_valid(self.gage_ids, self.obs_reader.gage_dict)
            log.info(f"Filtered {n} gages with DA_VALID=False")
        elif self.cfg.experiment.max_area_diff_sqkm is not None:
            self.gage_ids, n = filter_gages_by_area_threshold(
                self.gage_ids, self.obs_reader.gage_dict, self.cfg.experiment.max_area_diff_sqkm
            )
        self.gages_adjacency = read_zarr(Path(self.cfg.data_sources.gages_adjacency))
        self.gage_ids, n_hw = filter_headwater_gages(self.gage_ids, self.gages_adjacency)
        log.info(f"Training: {len(self.gage_ids)} gauged locations ({n_hw} headwaters removed)")

    def _init_inference(self) -> None:
        if self.cfg.data_sources.target_catchments is not None:
            self.network_graph, self.hf_id_to_node, _ = _build_network_graph(self.conus_adjacency)
            self.routing_dataclass = self._build_routing_data_target_catchments()
        elif self.cfg.data_sources.gages is not None and self.cfg.data_sources.gages_adjacency is not None:
            self.obs_reader = IcechunkUSGSReader(cfg=self.cfg)
            self.observations = self.obs_reader.read_data(dates=self.dates)
            self.gage_ids = np.array([str(_id.zfill(8)) for _id in self.obs_reader.gage_dict["STAID"]])
            self.gages_adjacency = read_zarr(Path(self.cfg.data_sources.gages_adjacency))
            self.gage_ids, _ = filter_headwater_gages(self.gage_ids, self.gages_adjacency)
            self.routing_dataclass = self._build_routing_data_gages()
        else:
            self.routing_dataclass = self._build_routing_data_all_catchments()

    def _collate_gages(self, batch: np.ndarray) -> RoutingDataclass:
        valid_mask = np.isin(batch, list(self.gages_adjacency.keys()))
        batch = batch[valid_mask].tolist()
        coo, _gage_idx, gage_catchment = construct_network_matrix(batch, self.gages_adjacency)
        edge_indices = (
            np.unique(np.concatenate([coo.row, coo.col])) if coo.nnz > 0 else np.array([], dtype=int)
        )
        active_indices = np.unique(np.concatenate([edge_indices, np.array(_gage_idx, dtype=int)]))
        index_mapping = {orig: comp for comp, orig in enumerate(active_indices)}
        c_rows = np.array([index_mapping[i] for i in coo.row]) if coo.nnz > 0 else np.array([], dtype=int)
        c_cols = np.array([index_mapping[i] for i in coo.col]) if coo.nnz > 0 else np.array([], dtype=int)
        N = len(active_indices)
        compressed_csr = sparse.coo_matrix((coo.data[: len(c_rows)], (c_rows, c_cols)), shape=(N, N)).tocsr()
        compressed_ids = self.merit_ids[active_indices]
        compressed_fp = self.flowpath_attr.reindex(compressed_ids)
        outflow_idx = []
        for _idx in _gage_idx:
            mask = np.isin(coo.row, _idx)
            orig_cols = coo.col[np.where(mask)[0]]
            outflow_idx.append(
                np.array([index_mapping[c] for c in orig_cols])
                if len(orig_cols) > 0
                else np.array([index_mapping[int(_idx)]])
            )
        flow_scale = build_flow_scale_tensor(
            batch=batch,
            gage_dict=self.obs_reader.gage_dict,
            gage_compressed_indices=[index_mapping[int(i)] for i in _gage_idx],
            num_segments=N,
        )
        adj, sa, nsa, fpt = self._build_common_tensors(compressed_csr, compressed_ids, compressed_fp)
        obs = create_hydrofabric_observations(
            dates=self.dates, gage_ids=np.array(batch), observations=self.observations
        )
        return RoutingDataclass(
            spatial_attributes=sa,
            length=fpt["length"],
            slope=fpt["slope"],
            side_slope=fpt["side_slope"],
            top_width=fpt["top_width"],
            x=fpt["x"],
            dates=self.dates,
            adjacency_matrix=adj,
            normalized_spatial_attributes=nsa,
            observations=obs,
            divide_ids=compressed_ids,
            outflow_idx=outflow_idx,
            gage_catchment=gage_catchment,
            flow_scale=flow_scale,
        )

---
## 7. Using your custom dataset in a training or routing run

The built-in datasets are accessed via the `GeoDataset` enum:

```python
dataset = cfg.geodataset.get_dataset_class(cfg=cfg)  # returns Merit or LynkerHydrofabric
```

For a custom subclass you instantiate it directly and pass it to the same DataLoader pattern used by the training script:

In [ ]:
import torch
from torch.utils.data import DataLoader, RandomSampler

from ddr.validation.configs import validate_config

# Load your config — adjust the config name to match your YAML file
cfg = validate_config(config_name="merit_training")

# Instantiate your custom dataset — same signature as the built-in classes
dataset = Merit(cfg=cfg)  # replace Merit with your class

# The DataLoader uses the shared collate_fn from BaseGeoDataset
sampler = RandomSampler(dataset, generator=torch.Generator().manual_seed(cfg.seed))
dataloader = DataLoader(
    dataset=dataset,
    batch_size=cfg.experiment.batch_size,
    num_workers=0,
    sampler=sampler,
    collate_fn=dataset.collate_fn,
    drop_last=True,
)

# Each iteration yields a RoutingDataclass ready for the routing engine
for routing_dataclass in dataloader:
    print("Adjacency shape  :", routing_dataclass.adjacency_matrix.shape)
    print("Attributes shape :", routing_dataclass.normalized_spatial_attributes.shape)
    print("Divide IDs       :", routing_dataclass.divide_ids[:5])
    break

---
## 8. Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| Not running `ddr-engine` before training | `FileNotFoundError` on the zarr path in `read_zarr()` | Run the engine build command first (section 3.3) |
| Engine built with wrong gage CSV | `self.gages_adjacency` has no key for valid gages; all batches are empty after the validity mask | Rebuild with the correct `--gages` CSV |
| `_get_attributes` returns `(N, num_attrs)` instead of `(num_attrs, N)` | Shape mismatch in `_build_common_tensors` normalisation broadcast | Transpose before returning |
| Forgetting `.T` on `normalized_spatial_attributes` | KAN receives wrong shape; silent wrong result | Add `.T` after z-scoring |
| `length` in km instead of metres | MC Courant number is 1000× too small; routing is numerically unstable | Multiply `lengthkm` by 1000 in `_build_common_tensors` |
| `_init_training` does not set `self.obs_reader` | `AttributeError` in `_collate_gages` when `build_flow_scale_tensor` accesses `gage_dict` | Set `self.obs_reader = IcechunkUSGSReader(cfg=self.cfg)` |
| `_init_inference` does not set `self.routing_dataclass` | `AssertionError: No RoutingDataclass, cannot batch` from base class `collate_fn` | Build and assign at the end of each inference branch |
| `top_width` / `side_slope` left as `None` instead of `torch.empty(0)` | `TypeError` in routing engine geometry branch selection | Set unused geometry tensors to `torch.empty(0)` |